# personal.ipynb

Ноутбук собирает единый реестр персонала из трёх Excel-файлов и строит витрины ресурсов по срокам:

- **t** — ресурсы на дату `D`
- **d+1** — ресурсы на дату `D+1`, показанные в строке даты `D`
- **d+7** — ресурсы на дату `D+7`, показанные в строке даты `D`
- **Факт** — фактическая численность на дату `D`
- **План_максимальный** — максимальное количество ресурсов в пределах года для той же группы

Важный момент по логике:

- `Ресурсы_t`, `Ресурсы_d+1`, `Ресурсы_d+7` и `План_максимальный` считаются **из одного и того же источника**:
  `План/Факт == "Ресурсы для выполнения программы"`.
- `План_максимальный` — это **максимум дневного количества ресурсов** по группе в пределах года.
- `Факт` считается отдельно из строк, где в колонке `План/Факт` есть слово **"факт"**.

То есть `План_максимальный` берётся **оттуда же**, откуда `t`, `d+1`, `d+7`.


In [16]:
from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------
# НАСТРОЙКИ
# ------------------------------

# Папка с исходными Excel-файлами.
SOURCE_DIR = Path("source")

# Имена файлов.
INITIAL_PLAN_FILE = "Первоначальный план.xlsx"
PLAN_COUNT_FILE = "План_численность.xlsx"
FACT_COUNT_FILE = "Факт_численность.xlsx"

# Имя листа.
SHEET_NAME = "Лист1"

# Основной источник ресурсов.
RESOURCE_LABEL = "Ресурсы для выполнения программы"

# По каким словам искать факт в колонке "План/Факт".
# Если после расчёта "Факт" везде нулевой, проверь source_counts
# и подправь этот список.
FACT_CONTAINS = ("факт",)

# Русские названия месяцев.
RU_MONTHS = {
    1: "январь",
    2: "февраль",
    3: "март",
    4: "апрель",
    5: "май",
    6: "июнь",
    7: "июль",
    8: "август",
    9: "сентябрь",
    10: "октябрь",
    11: "ноябрь",
    12: "декабрь",
}


In [17]:
# ------------------------------
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ------------------------------

def _to_datetime_safe(df: pd.DataFrame, columns) -> pd.DataFrame:
    """
    Безопасно приводит существующие колонки к datetime.
    """
    df = df.copy()
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df


def _expand_dates(start, end):
    """
    Возвращает список дат от start до end включительно.

    Используется для разворота периода работы сотрудника
    в ежедневные строки.
    """
    if pd.isna(start) or pd.isna(end):
        return []
    if end < start:
        return []

    return pd.date_range(start=start, end=end, freq="D").date.tolist()


def _normalize_text_columns(df: pd.DataFrame, cols) -> pd.DataFrame:
    """
    Приводит указанные текстовые колонки к типу string и убирает пробелы по краям.
    """
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = df[col].astype("string").str.strip()
    return df


def _contains_any(series: pd.Series, patterns) -> pd.Series:
    """
    Возвращает булеву маску:
    True, если значение строки содержит хотя бы один шаблон из patterns.
    """
    s = series.astype("string").fillna("").str.lower()
    mask = pd.Series(False, index=series.index)
    for pattern in patterns:
        mask = mask | s.str.contains(str(pattern).lower(), regex=False)
    return mask


def _daily_count(df: pd.DataFrame, group_cols, value_name: str) -> pd.DataFrame:
    """
    Считает количество строк по дате и группам.
    """
    return (
        df.groupby(["Дата"] + group_cols, dropna=False)
        .size()
        .reset_index(name=value_name)
    )


In [18]:
# ------------------------------
# 1. ПЕРВОНАЧАЛЬНЫЙ ПЛАН
# ------------------------------

def transform_initial_plan(df: pd.DataFrame) -> pd.DataFrame:
    """
    Преобразует файл "Первоначальный план" в единый формат.

    Логика повторяет Power Query:
    - приводит текстовые поля;
    - приводит даты;
    - разворачивает период работы в ежедневные строки;
    - переименовывает колонки под общий реестр;
    - оставляет только 2026 год.
    """
    df = df.copy()

    # Текстовые поля.
    text_cols = [
        "Подразделение",
        "Источник",
        "ЦФО",
        "Должность",
        "ФИО ",
    ]
    df = _normalize_text_columns(df, text_cols)

    # Даты приема/увольнения.
    if "Дата.приема.2026" in df.columns:
        df["Дата.приема.2026"] = pd.to_datetime(df["Дата.приема.2026"], errors="coerce").dt.date

    if "Дата.увольнения.2026" in df.columns:
        df["Дата.увольнения.2026"] = pd.to_datetime(df["Дата.увольнения.2026"], errors="coerce").dt.date

    # Разворачиваем период работы в список дат.
    df["Дата"] = df.apply(
        lambda row: _expand_dates(row["Дата.приема.2026"], row["Дата.увольнения.2026"]),
        axis=1,
    )

    # Превращаем список дат в отдельные строки.
    df = df.explode("Дата", ignore_index=True)
    df = df[df["Дата"].notna()].copy()
    df["Дата"] = pd.to_datetime(df["Дата"], errors="coerce").dt.date

    # Переименовываем колонки под общий формат.
    df = df.rename(
        columns={
            "Дата.приема.2026": "Дата начала работы",
            "Дата.увольнения.2026": "Дата окончания работы",
            "Источник": "План/Факт",
            "ФИО ": "ФИО",
            "Должность": "Профессия",
        }
    )

    # Оставляем только 2026 год.
    df = df[pd.to_datetime(df["Дата"], errors="coerce").dt.year == 2026].copy()

    return df


In [19]:
# ------------------------------
# 2. ПЛАН / ФАКТ ПО ЧИСЛЕННОСТИ
# ------------------------------

def transform_staff_count(df: pd.DataFrame) -> pd.DataFrame:
    """
    Универсальная функция для файлов:
    - План_численность.xlsx
    - Факт_численность.xlsx

    Логика повторяет Power Query:
    - приводит типы;
    - разворачивает период [Дата начала работы] -> [Дата окончания работы]
      в ежедневные строки;
    - удаляет лишние колонки;
    - приводит названия колонок к общему формату.
    """
    df = df.copy()

    # Текстовые поля.
    text_cols = [
        "План/Факт",
        "Подразделение",
        "Отдел",
        "ФИО работника",
        "Должность/профессияработника",
    ]
    df = _normalize_text_columns(df, text_cols)

    # Даты.
    if "Дата начала работы" in df.columns:
        df["Дата начала работы"] = pd.to_datetime(df["Дата начала работы"], errors="coerce").dt.date

    if "Дата окончания работы" in df.columns:
        df["Дата окончания работы"] = pd.to_datetime(df["Дата окончания работы"], errors="coerce").dt.date

    # Разворачиваем период в ежедневные записи.
    df["Дата"] = df.apply(
        lambda row: _expand_dates(row["Дата начала работы"], row["Дата окончания работы"]),
        axis=1,
    )

    df = df.explode("Дата", ignore_index=True)
    df = df[df["Дата"].notna()].copy()
    df["Дата"] = pd.to_datetime(df["Дата"], errors="coerce").dt.date

    # Удаляем лишнее.
    df.drop(columns=["Год", "Пол", "Основная профессия при приеме"], errors="ignore", inplace=True)

    # Переименовываем колонки под единый формат.
    df = df.rename(
        columns={
            "Отдел": "ЦФО",
            "ФИО работника": "ФИО",
            "Должность/профессияработника": "Профессия",
        }
    )

    return df


In [20]:
# ------------------------------
# 3. ЕДИНЫЙ РЕЕСТР
# ------------------------------

def transform_registry(
    df_initial_plan: pd.DataFrame,
    df_plan_count: pd.DataFrame,
    df_fact_count: pd.DataFrame,
) -> pd.DataFrame:
    """
    Собирает единый реестр из трёх подготовленных источников.

    Что делает функция:
    1. Объединяет три таблицы.
    2. Оставляет только 2025 и 2026 годы.
    3. Нормализует подразделения.
    4. Добавляет служебные колонки: Год, ДеньМесяц, Месяц.
    5. Нормализует названия профессий.
    """
    df = pd.concat(
        [df_initial_plan, df_plan_count, df_fact_count],
        ignore_index=True,
        sort=False,
    ).copy()

    # Приводим даты к datetime.
    df = _to_datetime_safe(df, ["Дата", "Дата начала работы", "Дата окончания работы"])

    # Оставляем только нужные годы.
    df["_ГодФильтр"] = df["Дата"].dt.year
    df = df[df["_ГодФильтр"].isin([2025, 2026])].copy()

    # Удаляем временные поля.
    df.drop(columns=["Примечание", "_ГодФильтр"], errors="ignore", inplace=True)

    # Нормализуем подразделения.
    if "Подразделение" in df.columns:
        df["Подразделение"] = df["Подразделение"].astype("string")
        df["Подразделение"] = df["Подразделение"].str.replace(
            "Административно-хозяйственный отдел", "Администрация", regex=False
        )
        df["Подразделение"] = df["Подразделение"].str.replace(
            "Участок ", "", regex=False
        )
        df["Подразделение"] = df["Подразделение"].str.replace(
            "Эрел", "Хаптагай-Хая", regex=False
        )
        df["Подразделение"] = df["Подразделение"].str.strip()

    # Нормализуем профессии.
    if "Профессия" in df.columns:
        df["Профессия"] = df["Профессия"].astype("string").str.strip()

        prof_replace = {
            "Электрослесарь (слесарь) дежурный и по ремонту оборудования": "Электрослесарь дежурный по ремонту оборудования",
            "Слесарь ТО": "Слесарь по техническому обслуживанию",
            "Водитель погрузчика": "Машинист погрузчика",
            "Мастер горный": "Горный мастер",
            "Электрогазосварщик, занятый на резке и ручной сварке": "Электрогазосварщик",
            "Комендант общежития": "Комендант",
            "Водитель автомобиля - экспедитор": "Водитель-экспедитор",
            "Слесарь на монтаже промывочных установок": "Слесарь-монтажник промывочного оборудования",
            "Слесарь дежурный по ремонту оборудования": "Слесарь",
            "Машинист колесного бульдозера": "Машинист погрузчика",
            "Машинист буровой установки POC": "Машинист буровой установки D55",
        }
        df["Профессия"] = df["Профессия"].replace(prof_replace)

    # Нормализуем ФИО и План/Факт.
    for col in ["ФИО", "План/Факт", "ЦФО"]:
        if col in df.columns:
            df[col] = df[col].astype("string").str.strip()

    # Служебные колонки.
    df["Год"] = df["Дата"].dt.year
    df["ДеньМесяц"] = df["Дата"].dt.strftime("%d.%m")
    df["Месяц"] = df["Дата"].dt.month.map(RU_MONTHS)

    # Формируем КлючУчета, как в Power Query, а затем удаляем.
    fio = (
        df["ФИО"].astype("string").fillna("")
        if "ФИО" in df.columns
        else pd.Series("", index=df.index, dtype="string")
    )
    подразделение = (
        df["Подразделение"].astype("string").fillna("")
        if "Подразделение" in df.columns
        else pd.Series("", index=df.index, dtype="string")
    )
    профессия = (
        df["Профессия"].astype("string").fillna("")
        if "Профессия" in df.columns
        else pd.Series("", index=df.index, dtype="string")
    )
    дата_начала = (
        pd.to_datetime(df["Дата начала работы"], errors="coerce").dt.strftime("%Y%m%d").fillna("")
        if "Дата начала работы" in df.columns
        else pd.Series("", index=df.index, dtype="string")
    )
    дата_окончания = (
        pd.to_datetime(df["Дата окончания работы"], errors="coerce").dt.strftime("%Y%m%d").fillna("")
        if "Дата окончания работы" in df.columns
        else pd.Series("", index=df.index, dtype="string")
    )

    df["КлючУчета"] = np.where(
        fio == "Вакансия",
        подразделение + "|" + профессия + "|" + дата_начала + "|" + дата_окончания,
        fio,
    )

    # Удаляем временный ключ, как в M-коде.
    df.drop(columns=["КлючУчета"], errors="ignore", inplace=True)

    # Приводим Id к Int64, если колонка есть.
    if "Id" in df.columns:
        df["Id"] = pd.to_numeric(df["Id"], errors="coerce").astype("Int64")

    return df


In [21]:
# ------------------------------
# 4. ВИТРИНА РЕСУРСОВ ПО ГОРИЗОНТАМ
# ------------------------------

def build_resources_horizon(
    df: pd.DataFrame,
    group_cols=None,
    resource_label: str = RESOURCE_LABEL,
    fact_contains=FACT_CONTAINS,
) -> pd.DataFrame:
    """
    Строит витрину ресурсов по горизонтам.

    На выходе:
    - Ресурсы_t
    - Ресурсы_d+1
    - Ресурсы_d+7
    - Факт
    - План_максимальный

    Важно:
    - Ресурсы_t / d+1 / d+7 и План_максимальный считаются из одного источника:
      План/Факт == resource_label
    - План_максимальный = максимум дневного количества ресурсов в пределах года
      для той же группы
    - Факт берётся из строк, где План/Факт содержит одно из слов из fact_contains
    """
    if group_cols is None:
        group_cols = ["Подразделение"]

    work = df.copy()
    work["Дата"] = pd.to_datetime(work["Дата"], errors="coerce").dt.normalize()
    work = _normalize_text_columns(work, group_cols + ["План/Факт"])

    # 1. Берём только ресурсные строки.
    resource_df = work[work["План/Факт"].eq(resource_label)].copy()

    # Если ресурсных строк нет, возвращаем пустую таблицу нужной структуры.
    if resource_df.empty:
        result_cols = ["Дата"] + group_cols + [
            "Ресурсы_t",
            "Ресурсы_d+1",
            "Ресурсы_d+7",
            "Факт",
            "План_максимальный",
            "ДеньМесяц",
            "Месяц",
            "Год",
        ]
        return pd.DataFrame(columns=result_cols)

    # 2. Ресурсы на дату D.
    t = _daily_count(resource_df, group_cols, "Ресурсы_t")

    # 3. Ресурсы на D+1, показанные в строке даты D.
    # Если ресурс есть на 17.04, он должен попасть в d+1 на 16.04,
    # поэтому здесь используется -1 день.
    d1_source = resource_df.copy()
    d1_source["Дата"] = d1_source["Дата"] - pd.Timedelta(days=1)
    d1 = _daily_count(d1_source, group_cols, "Ресурсы_d+1")

    # 4. Ресурсы на D+7, показанные в строке даты D.
    d7_source = resource_df.copy()
    d7_source["Дата"] = d7_source["Дата"] - pd.Timedelta(days=7)
    d7 = _daily_count(d7_source, group_cols, "Ресурсы_d+7")

    # 5. Склеиваем горизонты.
    result = (
        t.merge(d1, on=["Дата"] + group_cols, how="outer")
         .merge(d7, on=["Дата"] + group_cols, how="outer")
    )

    # 6. Факт на дату D.
    fact_mask = _contains_any(work["План/Факт"], fact_contains)
    fact_df = work[fact_mask].copy()

    if not fact_df.empty:
        fact_daily = _daily_count(fact_df, group_cols, "Факт")
        result = result.merge(fact_daily, on=["Дата"] + group_cols, how="left")
    else:
        result["Факт"] = 0

    # 7. План_максимальный считаем из того же ресурса, что и t / d+1 / d+7.
    resource_daily_for_max = _daily_count(resource_df, group_cols, "Ресурсы_на_дату")
    resource_daily_for_max["Год"] = pd.to_datetime(resource_daily_for_max["Дата"]).dt.year

    plan_max = (
        resource_daily_for_max
        .groupby(["Год"] + group_cols, dropna=False)["Ресурсы_на_дату"]
        .max()
        .reset_index(name="План_максимальный")
    )

    result["Год"] = pd.to_datetime(result["Дата"]).dt.year
    result = result.merge(plan_max, on=["Год"] + group_cols, how="left")

    # 8. Дополнительные поля и типы.
    for col in ["Ресурсы_t", "Ресурсы_d+1", "Ресурсы_d+7", "Факт", "План_максимальный"]:
        if col not in result.columns:
            result[col] = 0
        result[col] = result[col].fillna(0).astype(int)

    result["ДеньМесяц"] = pd.to_datetime(result["Дата"], errors="coerce").dt.strftime("%d.%m")
    result["Месяц"] = pd.to_datetime(result["Дата"], errors="coerce").dt.month.map(RU_MONTHS)

    front_cols = (
        ["Дата", "ДеньМесяц", "Месяц", "Год"]
        + group_cols
        + ["Ресурсы_t", "Ресурсы_d+1", "Ресурсы_d+7", "Факт", "План_максимальный"]
    )
    other_cols = [c for c in result.columns if c not in front_cols]
    result = result[front_cols + other_cols]

    return result.sort_values(["Дата"] + group_cols).reset_index(drop=True)


In [22]:
# ------------------------------
# 5. ЗАГРУЗКА ИСХОДНЫХ ДАННЫХ
# ------------------------------

initial_plan_path = SOURCE_DIR / INITIAL_PLAN_FILE
plan_count_path = SOURCE_DIR / PLAN_COUNT_FILE
fact_count_path = SOURCE_DIR / FACT_COUNT_FILE

print("Рабочая папка:", Path.cwd())
print("Папка source:", SOURCE_DIR.resolve())
print()

if SOURCE_DIR.exists():
    print("Файлы в source:")
    for file in sorted(SOURCE_DIR.glob("*")):
        print(" -", file.name)
else:
    print("Папка source не найдена.")

print()
print("Проверка путей:")
print("Первоначальный_план:", initial_plan_path, "->", initial_plan_path.exists())
print("План_численность  :", plan_count_path, "->", plan_count_path.exists())
print("Факт_численность  :", fact_count_path, "->", fact_count_path.exists())

# Читаем исходные Excel-файлы.
df_initial_plan_raw = pd.read_excel(initial_plan_path, sheet_name=SHEET_NAME, engine="openpyxl")
df_plan_count_raw = pd.read_excel(plan_count_path, sheet_name=SHEET_NAME, engine="openpyxl")
df_fact_count_raw = pd.read_excel(fact_count_path, sheet_name=SHEET_NAME, engine="openpyxl")

# Подготавливаем каждую таблицу отдельно.
df_initial_plan = transform_initial_plan(df_initial_plan_raw)
df_plan_count = transform_staff_count(df_plan_count_raw)
df_fact_count = transform_staff_count(df_fact_count_raw)

print()
print("Размеры после подготовки:")
print("df_initial_plan:", df_initial_plan.shape)
print("df_plan_count  :", df_plan_count.shape)
print("df_fact_count  :", df_fact_count.shape)


Рабочая папка: C:\Users\poisk-12\PycharmProjects\Personals
Папка source: C:\Users\poisk-12\PycharmProjects\Personals\source

Файлы в source:
 - Первоначальный план.xlsx
 - План_численность.xlsx
 - Факт_численность.xlsx

Проверка путей:
Первоначальный_план: source\Первоначальный план.xlsx -> True
План_численность  : source\План_численность.xlsx -> True
Факт_численность  : source\Факт_численность.xlsx -> True

Размеры после подготовки:
df_initial_plan: (131792, 9)
df_plan_count  : (114397, 12)
df_fact_count  : (325670, 8)


In [23]:
# ------------------------------
# 6. СБОРКА РЕЕСТРА
# ------------------------------

registry_df = transform_registry(
    df_initial_plan=df_initial_plan,
    df_plan_count=df_plan_count,
    df_fact_count=df_fact_count,
)

print("Размер реестра:", registry_df.shape)

# Посмотри, какие реальные значения есть в колонке "План/Факт".
# Это полезно для проверки факта.
source_counts = registry_df["План/Факт"].value_counts(dropna=False)
display(source_counts)

display(registry_df.head(10))


Размер реестра: (571803, 15)


План/Факт
Факт                                325670
Ресурсы для выполнения программы    131792
План                                114341
Name: count, dtype: Int64

,Подразделение,План/Факт,ЦФО,Профессия,ФИО,Дата начала работы,Дата окончания работы,Дата,Дата увольнения?,Дата звонка работнику,Подтверждение приезда работником,Подтверждение принял,Год,ДеньМесяц,Месяц
0,Администрация,Ресурсы для выполнения программы,Строительный отдел,Плотник,Ворзуль Максим Сергеевич,2026-02-25,2026-11-25,2026-02-25,NaN,NaT,NaN,NaN,2026,25.02,февраль
1,Администрация,Ресурсы для выполнения программы,Строительный отдел,Плотник,Ворзуль Максим Сергеевич,2026-02-25,2026-11-25,2026-02-26,NaN,NaT,NaN,NaN,2026,26.02,февраль
2,Администрация,Ресурсы для выполнения программы,Строительный отдел,Плотник,Ворзуль Максим Сергеевич,2026-02-25,2026-11-25,2026-02-27,NaN,NaT,NaN,NaN,2026,27.02,февраль
3,Администрация,Ресурсы для выполнения программы,Строительный отдел,Плотник,Ворзуль Максим Сергеевич,2026-02-25,2026-11-25,2026-02-28,NaN,NaT,NaN,NaN,2026,28.02,февраль
4,Администрация,Ресурсы для выполнения программы,Строительный отдел,Плотник,Ворзуль Максим Сергеевич,2026-02-25,2026-11-25,2026-03-01,NaN,NaT,NaN,NaN,2026,01.03,март
5,Администрация,Ресурсы для выполнения программы,Строительный отдел,Плотник,Ворзуль Максим Сергеевич,2026-02-25,2026-11-25,2026-03-02,NaN,NaT,NaN,NaN,2026,02.03,март
6,Администрация,Ресурсы для выполнения программы,Строительный отдел,Плотник,Ворзуль Максим Сергеевич,2026-02-25,2026-11-25,2026-03-03,NaN,NaT,NaN,NaN,2026,03.03,март
7,Администрация,Ресурсы для выполнения программы,Строительный отдел,Плотник,Ворзуль Максим Сергеевич,2026-02-25,2026-11-25,2026-03-04,NaN,NaT,NaN,NaN,2026,04.03,март
8,Администрация,Ресурсы для выполнения программы,Строительный отдел,Плотник,Ворзуль Максим Сергеевич,2026-02-25,2026-11-25,2026-03-05,NaN,NaT,NaN,NaN,2026,05.03,март
9,Администрация,Ресурсы для выполнения программы,Строительный отдел,Плотник,Ворзуль Максим Сергеевич,2026-02-25,2026-11-25,2026-03-06,NaN,NaT,NaN,NaN,2026,06.03,март


In [24]:
# ------------------------------
# 7. ВИТРИНЫ РЕСУРСОВ
# ------------------------------

# 1) По подразделению.
resources_by_department_df = build_resources_horizon(
    registry_df,
    group_cols=["Подразделение"],
)

# 2) По подразделению и профессии.
resources_by_department_prof_df = build_resources_horizon(
    registry_df,
    group_cols=["Подразделение", "Профессия"],
)

# 3) По подразделению, профессии и сотруднику.
resources_by_employee_df = build_resources_horizon(
    registry_df,
    group_cols=["Подразделение", "Профессия", "ФИО"],
)

print("Ресурсы по подразделениям          :", resources_by_department_df.shape)
print("Ресурсы по подразделению/профессии :", resources_by_department_prof_df.shape)
print("Ресурсы по сотрудникам             :", resources_by_employee_df.shape)

display(resources_by_employee_df.head(20))


Ресурсы по подразделениям          : (4533, 10)
Ресурсы по подразделению/профессии : (63380, 11)
Ресурсы по сотрудникам             : (118263, 12)


,Дата,ДеньМесяц,Месяц,Год,Подразделение,Профессия,ФИО,Ресурсы_t,Ресурсы_d+1,Ресурсы_d+7,Факт,План_максимальный
0,2025-12-25,25.12,декабрь,2025,Администрация,Бухгалтер по налоговому учету,Вожакова Ирина Юрьевна,0,0,1,0,0
1,2025-12-25,25.12,декабрь,2025,Администрация,Генеральный директор,Каздобин Андрей Витальевич,0,0,1,1,0
2,2025-12-25,25.12,декабрь,2025,Администрация,Главный инженер,Баранов Игорь Александрович,0,0,1,1,0
3,2025-12-25,25.12,декабрь,2025,Администрация,Директор по развитию минерально-сырьевой базы,Рыбаков Валерий Юрьевич,0,0,1,1,0
4,2025-12-25,25.12,декабрь,2025,Администрация,Заведующий отделом,Пряхина Татьяна Ивановна,0,0,1,1,0
5,2025-12-25,25.12,декабрь,2025,Администрация,Заместитель генерального директора по безопасн...,Марченков Сергей Михайлович,0,0,1,1,0
6,2025-12-25,25.12,декабрь,2025,Администрация,Менеджер по персоналу,Калашникова Елена Юрьевна,0,0,1,0,0
7,2025-12-25,25.12,декабрь,2025,Администрация,Менеджер по снабжению,Серебряков Роман Владимирович,0,0,1,1,0
8,2025-12-25,25.12,декабрь,2025,Администрация,Начальник отдела,Булгаков Владимир Вячеславович,0,0,1,1,0
9,2025-12-25,25.12,декабрь,2025,Администрация,Начальник отдела,Иванов Игорь Николаевич,0,0,1,1,0


In [25]:
# ------------------------------
# 8. БЫСТРЫЕ ПРОВЕРКИ
# ------------------------------

display(
    resources_by_employee_df[
        [
            "Дата",
            "Подразделение",
            "Профессия",
            "ФИО",
            "Ресурсы_t",
            "Ресурсы_d+1",
            "Ресурсы_d+7",
            "Факт",
            "План_максимальный",
        ]
    ].head(20)
)

print("Нулей в Факт:", int((resources_by_employee_df["Факт"] == 0).sum()), "из", len(resources_by_employee_df))


,Дата,Подразделение,Профессия,ФИО,Ресурсы_t,Ресурсы_d+1,Ресурсы_d+7,Факт,План_максимальный
0,2025-12-25,Администрация,Бухгалтер по налоговому учету,Вожакова Ирина Юрьевна,0,0,1,0,0
1,2025-12-25,Администрация,Генеральный директор,Каздобин Андрей Витальевич,0,0,1,1,0
2,2025-12-25,Администрация,Главный инженер,Баранов Игорь Александрович,0,0,1,1,0
3,2025-12-25,Администрация,Директор по развитию минерально-сырьевой базы,Рыбаков Валерий Юрьевич,0,0,1,1,0
4,2025-12-25,Администрация,Заведующий отделом,Пряхина Татьяна Ивановна,0,0,1,1,0
5,2025-12-25,Администрация,Заместитель генерального директора по безопасн...,Марченков Сергей Михайлович,0,0,1,1,0
6,2025-12-25,Администрация,Менеджер по персоналу,Калашникова Елена Юрьевна,0,0,1,0,0
7,2025-12-25,Администрация,Менеджер по снабжению,Серебряков Роман Владимирович,0,0,1,1,0
8,2025-12-25,Администрация,Начальник отдела,Булгаков Владимир Вячеславович,0,0,1,1,0
9,2025-12-25,Администрация,Начальник отдела,Иванов Игорь Николаевич,0,0,1,1,0


Нулей в Факт: 83780 из 118263


In [26]:
# ------------------------------
# 9. СОХРАНЕНИЕ РЕЗУЛЬТАТОВ
# ------------------------------

output_path = Path("Реестр_и_ресурсы.xlsx")

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    registry_df.to_excel(writer, sheet_name="Реестр", index=False)
    resources_by_department_df.to_excel(writer, sheet_name="Ресурсы_подразделения", index=False)
    resources_by_department_prof_df.to_excel(writer, sheet_name="Ресурсы_подр_проф", index=False)
    resources_by_employee_df.to_excel(writer, sheet_name="Ресурсы_по_людям", index=False)

print(f"Файл сохранён: {output_path.resolve()}")


Файл сохранён: C:\Users\poisk-12\PycharmProjects\Personals\Реестр_и_ресурсы.xlsx
